# Session 8 — Signal Processing: Sampling and Filtering

**Goal of this session:** understand sampling rate, then pull a buried oscillation back out of noise.

*Python for Neuroscience, session 8 of 12.*

## Why this matters

Raw electrophysiology is never clean. There is line noise at 50 or 60 Hz, slow drift from the electrode, muscle artefacts, and the thing you actually care about somewhere underneath.

Filtering is how you decide which frequencies to keep. It is the most-used and most-abused tool in the field, so it is worth understanding rather than copying.

## Sampling rate

Sampling rate is how many times per second you measured. Our toy signal is at 500 Hz, so the gap between samples is 1/500 of a second, which is 2 ms.

One rule governs everything: you can only recover frequencies below half your sampling rate. That half is called the Nyquist frequency. At 500 Hz you can see up to 250 Hz, and anything faster is lost or, worse, shows up disguised as something slower.

In [ ]:
import numpy as np


def generate_toy_signal(duration=2.0, sampling_rate=500.0, noise_level=0.5,
                        freq=10.0, amplitude=1.0, seed=0):
    """A toy oscillatory signal: one sine wave plus white noise.

    This is not a recording. It is a stand-in that behaves enough like an
    alpha rhythm to practise on. Returns the time axis and the signal.
    """
    rng = np.random.default_rng(seed)
    t = np.arange(0, duration, 1 / sampling_rate)
    signal = amplitude * np.sin(2 * np.pi * freq * t)
    signal = signal + noise_level * rng.standard_normal(t.size)
    return t, signal

In [ ]:
import numpy as np

sampling_rate = 500.0
t, clean = generate_toy_signal(duration=2.0, sampling_rate=sampling_rate,
                               noise_level=0.0, freq=10.0)

print(f"sampling rate    {sampling_rate} Hz")
print(f"time between samples {1 / sampling_rate * 1000:.1f} ms")
print(f"Nyquist frequency {sampling_rate / 2} Hz")
print(f"samples in 2 s    {len(t)}")

## Burying the signal

Now let's make life hard on purpose. Same 10 Hz oscillation, but with heavy noise and a slow drift added on top, which is what a real electrode gives you.

In [ ]:
rng = np.random.default_rng(1)

noise = 1.5 * rng.standard_normal(t.size)
drift = 2.0 * np.sin(2 * np.pi * 0.3 * t)          # slow 0.3 Hz drift
line = 0.8 * np.sin(2 * np.pi * 50.0 * t)          # 50 Hz mains hum

messy = clean + noise + drift + line
print("messy signal:", messy.shape)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(t, messy, linewidth=0.8, color="#718096")
ax.set_xlabel("time (s)", fontsize=13)
ax.set_ylabel("amplitude", fontsize=13)
ax.set_title("Raw signal: oscillation, noise, drift and mains hum", fontsize=15)
ax.tick_params(labelsize=12)
plt.tight_layout()
plt.show()

The 10 Hz rhythm is genuinely in there. You cannot see it.

## Designing a filter

We want to keep a band around 10 Hz and throw away everything else. That is a band-pass filter, and `scipy.signal` builds one in two steps.

First `butter()` designs the filter and gives you its coefficients. `Wn` is the band you want to keep, in Hz, and `fs` tells SciPy your sampling rate so it can do the conversion for you.

Then `filtfilt()` applies it. Use `filtfilt` rather than `lfilter`, because it runs the filter forwards and then backwards, which cancels out the time shift a filter would otherwise introduce. A shifted signal in a latency analysis is a wrong result.

In [ ]:
from scipy.signal import butter, filtfilt

b, a = butter(N=4, Wn=[8.0, 12.0], btype="bandpass", fs=sampling_rate)
filtered = filtfilt(b, a, messy)

print("filter order 4, band 8 to 12 Hz")
print("filtered:", filtered.shape)

`N=4` is the filter order, which controls how sharply it cuts. Higher is sharper and less stable. Between 2 and 6 covers almost everything you will need.

## Raw against filtered

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)

axes[0].plot(t, messy, linewidth=0.8, color="#718096")
axes[0].set_ylabel("amplitude", fontsize=13)
axes[0].set_title("Raw", fontsize=15)
axes[0].tick_params(labelsize=12)

axes[1].plot(t, filtered, linewidth=1.5, color="#2b6cb0", label="filtered")
axes[1].plot(t, clean, linewidth=1, color="crimson", linestyle="--",
             label="true 10 Hz signal")
axes[1].set_xlabel("time (s)", fontsize=13)
axes[1].set_ylabel("amplitude", fontsize=13)
axes[1].set_title("Band-pass filtered, 8 to 12 Hz", fontsize=15)
axes[1].tick_params(labelsize=12)
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.show()

The blue line tracks the red dashed line closely. We recovered an oscillation that was completely invisible in the panel above, and the red line is the ground truth we happen to know because we built the signal ourselves.

With real data you never have that red line. That is exactly why you should be careful.

## Looking at it in the frequency domain

A power spectrum answers "how much of each frequency is in here". `welch()` computes it, and it makes the filter's effect obvious.

In [ ]:
from scipy.signal import welch

f_raw, p_raw = welch(messy, fs=sampling_rate, nperseg=512)
f_filt, p_filt = welch(filtered, fs=sampling_rate, nperseg=512)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.semilogy(f_raw, p_raw, color="#718096", label="raw")
ax.semilogy(f_filt, p_filt, color="#2b6cb0", label="filtered")
ax.axvspan(8, 12, color="#2b6cb0", alpha=0.12, label="pass band")
ax.set_xlim(0, 70)
ax.set_xlabel("frequency (Hz)", fontsize=13)
ax.set_ylabel("power", fontsize=13)
ax.set_title("Power spectrum before and after filtering", fontsize=15)
ax.tick_params(labelsize=12)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

Two spikes in the grey line: one at 10 Hz, which is our signal, and one at 50 Hz, which is the mains. The blue line has kept the first and flattened everything else by orders of magnitude.

## One warning

A band-pass filter will produce a beautiful oscillation from pure noise if you ask it to. Filter white noise between 8 and 12 Hz and you get something that looks exactly like alpha. The rhythm you see after filtering is not evidence that the rhythm was there.

Always look at the unfiltered spectrum before you believe a filtered trace.

## Try it yourself

Set `noise_level` high, rebuild `messy` with no `clean` component at all, and run the same filter. Look at what comes out.

**Next session:** turning a plot into a statistical claim.